<a href="https://colab.research.google.com/github/joshbeckham-lab/SDFfixerVDS/blob/main/SDFfixerVDSmolssi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://raw.githubusercontent.com/joshbeckham-lab/imagefiles/main/virtual%20cures_RGB_orange_nickname.png" width="100" height="100" alt="Virtual Cures Logo" />

NAME: **SDF Fixer** GitHub Version

Started: 5/8/2026   Revised:  6/13/26   Dr. B

NOTE: this could be adapted for the EduPOD too

**PURPOSE:**
- to move compound name to top of the SDF info for each compound entry so that it will show up in PyMOL
- to fix the SDF breaks at the end of each compound with the four dollar symbols `$$$$`
- to fix the number of lines at the top of each entry before the X,Y,Z coordinates start so that they can be read in PyMOL

**From MOLSSI proposal:**
... a script in Python to operate on SDFs (structure data format) that our compounds are formatted in. Often these files are formatted incorrectly or could be improved for usability. The first task would be to use Python to edit these text files in batch to make modifications such as moving the name of the molecule to the top line so that it would appear in the PyMOL viewer when we visualize the docked compounds. This is a simple first step to coding that will allow students to become familiar with the environment and easily verify that everything is functioning correctly.


####INPUTS:
several individual SDF files for compounds that may have missing name on top line or that won't load in PyMOL properly.

####OUTPUTS:
an SDF file with multiple compounds all in one file along with the name on the top line.

**Language:** PyThon



**Attribution:**
some of the code and text blocks are modified from the PyBMB workshop in April of 2026 supported by NSF IUSE 2518733

##Saving Your Work

Colab auto-saves your notebook to Google Drive every few minutes, but you can also save manually.

Saving options:

1.   **Auto-save:** Happens automatically in the background. (but doesn't always work well. So, save manually too)
2. **Manual save:** Press **Ctrl + S** (Windows/Linux) or **Cmd + S** (Mac), or go to File → Save.
3.   **Save a copy in Drive:** File → Save a copy in Drive (creates a new file).
4.    **Download:** File → Download → Download .ipynb (saves a local copy to your computer).

All saved notebooks appear in your Google Drive under the Colab Notebooks folder. You can move the Colab file to a different location within Drive and not see a change in function.


##Navigation

Use the outline to the left to display headers in the document

<-- Click on the 3 dots and 3 lines icon at the left.


###Description of Steps:
- load dependencies
- upload your SDF file(s) to Google Colab folder (directory)
- read the SDF files, check if name is at top or not
- if name at top ask user if the want to change it by giving a prompt with that name already filled into the prompt
- if no name at top, then find the name of the ligand in the file and offer it as a suggestion while prompting the user to either:
  1. check if that suggested name is good or
  2. enter a new name to replace it.  
  If no name is found just prompt the user for the name.
- also check that the SDF file as four dollar symbols `$$$$` at the end of the file followed no carriage return at end.

Also need to check if 3 lines before the 'coordinates'
Then concatenate each of the files together
Output new name as "controls.sdf"



OR can the code make a prompt for the file upload?

NOTE:  when a single SDF file is missing the carriage return (line break) after the four dolar symbols `$$$$` that denotes the end of a compound, it won't show the compound name in PyMOL. However, if you add this, then the label on the top line will show up in PyMOL. For multiple compound SDFs, you do NOT want to have a carriage return (line break) at the end of the very last one.

For example, here are the ends of two SDF files:  

End of AspirinCID_2244x2_WorksButBadAtomCount.sdf:
...
> <PUBCHEM_COORDINATE_TYPE>
2
5
10

$$$$
<----carriage return is right here
----End of File----



End of AspirinCID_2244x2_WorksNoLastCarriage.sdf:
...
> <PUBCHEM_COORDINATE_TYPE>
2
5
10

$$$$ ----End of File----(no carriage return)


## PROMPT (to Gemini) to generate the PyThon code

Hi,  I need a Python script for Google Colabs that reads several SDF files and fixes them.
It should prompt the user to upload the input SDF files
Then checks if name is at top or not
- if no name at top, then find the name of the ligand in the file and offer it as a suggestion while prompting the user to either 1. check if that name is good or 2. enter a new name to replace it.  If no name is found just prompt the user for the name.
- also check that the SDF file as four dollar symbols `$$$$` at the end of the file followed no carriage return at end.

Also need to check if 3 lines before the 'coordinates' lines begin (3 header lines).  The coordinates are numbers describing the atoms.  If not -add lines or remove them as necessary.

Then concatenate each of the files together

Output new name as "controls.sdf"



# Gemini's Response
"This script is optimized for Google Colab. It uses the google.colab library to handle file uploads and downloads directly.

The logic identifies the Counts Line (the line containing V2000 or V3000) to anchor the file structure, ensuring that the atom coordinates are never shifted, while strictly enforcing the 3-line header and the `$$$$` terminator.


How this meets your requirements:"

    Strict Header Check: It ignores the original lines 2 and 3 and replaces them with standard placeholders. This ensures that the Counts Line is always exactly the 4th line, and the Coordinates start on the 5th line.

    Ligand Name Extraction: It intelligently checks if the top line is empty and scans the "Data Block" (lines starting with > <) to find a name to suggest to you.

    Terminator Fix: It uses .rstrip() to kill any carriage returns (\r) or trailing blank lines, ensuring the file ends cleanly with $$$$.

    Concatenation: It bundles every processed file into a single controls.sdf which is then automatically downloaded to your computer.

##Code to rename top line and fix errors in SDF.
Note: run this code cell below, then while it is running you have to scroll to the bottom of the code block -  where you will select files using the 'BROWSE' button and enter compound name information

In [ ]:
#@title Code to rename top line and fix errors in SDF  Version 2
import os
from google.colab import files

def process_and_merge_final_robust():
    print("Please upload the SDF files you want to fix and merge:")
    uploaded = files.upload()
    uploaded_filenames = list(uploaded.keys())

    if not uploaded_filenames:
        print("No files uploaded.")
        return

    all_processed_molecules = []

    for filename in uploaded_filenames:
        content_raw = uploaded[filename].decode('utf-8', errors='replace')
        lines = content_raw.splitlines()

        if not lines:
            continue

        fixes_made = []

        # 1. Find Anchor (Counts Line)
        counts_line_idx = -1
        for i, line in enumerate(lines):
            if "V2000" in line or "V3000" in line:
                counts_line_idx = i
                break

        if counts_line_idx == -1:
            print(f"\nCRITICAL ERROR: {filename} is not a valid SDF. Skipping.")
            continue

        # 2. Collect all potential info from the header block
        original_header_lines = [l.strip() for l in lines[:counts_line_idx]]

        # 3. Look for Name in Metadata Tags
        tag_name = ""
        for i, line in enumerate(lines):
            if line.startswith('> <') and any(tag in line.upper() for tag in ['NAME', 'IDENTIFIER', 'TITLE', 'CID']):
                if i + 1 < len(lines):
                    tag_name = lines[i+1].strip()
                    break

        # 4. Separate "Name" from "Metadata"
        # We look for the 'Software Signature' (contains -OEChem-, -RDKit-, etc.)
        detected_metadata = ""
        detected_name = ""

        # Check the header lines we found
        for h_line in original_header_lines:
            if not h_line: continue
            # Software signatures usually have hyphens or specific program names
            if "-" in h_line or "Chem" in h_line or "Babel" in h_line:
                detected_metadata = h_line
            elif not detected_name:
                detected_name = h_line

        # 5. Determine the "Current Name" to show the user
        # Priority: Tag Name > Header Name > Filename
        if tag_name:
            current_name = tag_name
            if tag_name != detected_name and detected_name:
                # If the header had a different name than the tag,
                # treat the header name as secondary metadata if line 2 is empty
                if not detected_metadata: detected_metadata = detected_name
        elif detected_name:
            current_name = detected_name
        else:
            current_name = filename.rsplit('.', 1)[0]

        # Diagnostic logs
        if counts_line_idx != 3:
            fixes_made.append(f"Header length adjusted (was {counts_line_idx} lines)")
        if tag_name and not detected_name:
            fixes_made.append("Name missing from top: Recovered from tags")

        # 6. User Prompt (The instructional style you like)
        print(f"\n" + "="*60)
        print(f"FILE: {filename}")
        if fixes_made:
            print("DIAGNOSTIC FIXES:")
            for fix in fixes_made:
                print(f"  [!] {fix}")
        print("-" * 60)
        print(f"Current Name: {current_name}")
        print("OPTIONS:")
        print("  - To KEEP: Press Enter")
        print("  - To ADD: Type '+' followed by text (e.g., +_control)")
        print("  - To REPLACE: Type the new name entirely")

        user_input = input("\nYour choice: ").strip()

        final_name = current_name
        if user_input:
            if user_input.startswith('+'):
                final_name = f"{current_name}{user_input[1:]}"
            else:
                final_name = user_input

        # 7. Reconstruct the Header Strictly
        # Line 1: Final Name
        # Line 2: Software Metadata (The -OEChem- info)
        # Line 3: Blank
        final_header = [
            final_name,
            detected_metadata if detected_metadata else "  Generated-v2",
            ""
        ]

        # 8. Assemble Molecule and Fix Terminator
        molecule_body = lines[counts_line_idx:]
        entry_text = "\n".join(final_header + molecule_body).rstrip()
        if not entry_text.endswith("$$$$"):
            entry_text += "\n$$$$"

        all_processed_molecules.append(entry_text)

    # 9. Final Save
    if all_processed_molecules:
        final_output = "\n".join(all_processed_molecules)
        output_name = "controls.sdf"
        with open(output_name, 'w', newline='\n') as f:
            f.write(final_output)

        print("\n" + "="*60)
        print(f"SUCCESS: Created '{output_name}' with {len(all_processed_molecules)} compounds.")
        files.download(output_name)

# Run
process_and_merge_final_robust()

In [ ]:
#@title Code to rename top line and fix errors in SDFVersion 3
import os
from google.colab import files

def process_and_merge_final_robust():
    print("Please upload the SDF files you want to fix and merge:")
    uploaded = files.upload()
    uploaded_filenames = list(uploaded.keys())

    if not uploaded_filenames:
        print("No files uploaded.")
        return

    all_processed_molecules = []

    for filename in uploaded_filenames:
        content_raw = uploaded[filename].decode('utf-8', errors='replace')
        lines = content_raw.splitlines()

        if not lines:
            continue

        fixes_made = []

        # 1. Find Anchor (Counts Line)
        counts_line_idx = -1
        for i, line in enumerate(lines):
            if "V2000" in line or "V3000" in line:
                counts_line_idx = i
                break

        if counts_line_idx == -1:
            print(f"\nCRITICAL ERROR: {filename} is not a valid SDF. Skipping.")
            continue

        # 2. Collect all potential info from the header block
        original_header_lines = [l.strip() for l in lines[:counts_line_idx]]

        # 3. Look for Name in Metadata Tags
        tag_name = ""
        for i, line in enumerate(lines):
            if line.startswith('> <') and any(tag in line.upper() for tag in ['NAME', 'IDENTIFIER', 'TITLE', 'CID']):
                if i + 1 < len(lines):
                    tag_name = lines[i+1].strip()
                    break

        # 4. Separate "Name" from "Metadata"
        detected_metadata = ""
        detected_name = ""

        for h_line in original_header_lines:
            if not h_line: continue
            if "-" in h_line or "Chem" in h_line or "Babel" in h_line:
                detected_metadata = h_line
            elif not detected_name:
                detected_name = h_line

        # 5. Determine the "Current Name" to show the user
        if tag_name:
            current_name = tag_name
            if tag_name != detected_name and detected_name:
                if not detected_metadata: detected_metadata = detected_name
        elif detected_name:
            current_name = detected_name
        else:
            current_name = filename.rsplit('.', 1)[0]

        # Diagnostic logs
        if counts_line_idx != 3:
            fixes_made.append(f"Header length adjusted (was {counts_line_idx} lines)")
        if tag_name and not detected_name:
            fixes_made.append("Name missing from top: Recovered from tags")

        # 6. User Prompt
        print(f"\n" + "="*60)
        print(f"FILE: {filename}")
        if fixes_made:
            print("DIAGNOSTIC FIXES:")
            for fix in fixes_made:
                print(f"  [!] {fix}")
        print("-" * 60)
        print(f"Current Name: {current_name}")
        print("OPTIONS:")
        print("  - To KEEP: Press Enter")
        print("  - To ADD: Type '+' followed by text (e.g., +_control)")
        print("  - To REPLACE: Type the new name entirely")

        user_input = input("\nYour choice: ").strip()

        final_name = current_name
        if user_input:
            if user_input.startswith('+'):
                final_name = f"{current_name}{user_input[1:]}"
            else:
                final_name = user_input

        # --- NEW CONSOLE OUTPUT LINE ---
        print(f"-> Final name used for top line: '{final_name}'")

        # 7. Reconstruct the Header Strictly
        final_header = [
            final_name,
            detected_metadata if detected_metadata else "  Generated-v2",
            ""
        ]

        # 8. Assemble Molecule and Fix Terminator
        molecule_body = lines[counts_line_idx:]
        entry_text = "\n".join(final_header + molecule_body).rstrip()
        if not entry_text.endswith("$$$$"):
            entry_text += "\n$$$$"

        all_processed_molecules.append(entry_text)

    # 9. Final Save
    if all_processed_molecules:
        final_output = "\n".join(all_processed_molecules)
        output_name = "controls.sdf"
        with open(output_name, 'w', newline='\n') as f:
            f.write(final_output)

        print("\n" + "="*60)
        print(f"SUCCESS: Created '{output_name}' with {len(all_processed_molecules)} compounds.")
        files.download(output_name)

# Run
process_and_merge_final_robust()

In [ ]:
#@title Code to clean the Contents folder of your temporary files after running
# run this cell if you want to re-do the script with some new compounds becuase
# the temporary files will be stored in the Google Colab
# and may interfere wiht next run
# note: to reset any variables, in Colab go to Edit >> Clear all outputs
import os
import shutil

# 1. Define the directory to clean (default Colab directory)
path = "/content/"

# 2. List of specific items to remove to keep the folder clean
# We target .sdf files and the 'vitamins' directory
files_to_remove = [f for f in os.listdir(path) if f.endswith('.sdf')]
folders_to_remove = ['vitamins'] # 'sample_data' is the Colab default folder

print("Cleaning workspace...")

# Remove files
for file in files_to_remove:
    os.remove(os.path.join(path, file))
    print(f"Deleted file: {file}")

# Remove folders
for folder in folders_to_remove:
    folder_path = os.path.join(path, folder)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Deleted folder: {folder}")

print("\nWorkspace is now clean! You're ready for a fresh start.")

Cleaning workspace...

Workspace is now clean! You're ready for a fresh start.


#End

# Alternative methods to load your SDF file(s) into the Google Colab

##  — Uploading data from file upload in Google Colab (Option 1: simplest)

Files can be uploaded to Colab from your local computer.


**Direct upload** - Click the folder icon at left, and drag or upload your file into the space.
Click on the 'content' folder, then drag and drop your files into this.  Or click the up arrow at top of this folders view "Upload to Session Storage"

**Note:** The files will only persist while you have this Google Colab open.  Files will disappear once you close (or the runtime goes out!!).
So, immediatelly save anything you want to keep
Each time you run this, you will need to re-upload the files.

**Note:** Data uploaded to Colab is not private and Colab is not compliant with US privacy laws such as FERPA or HIPAA. Do not upload data that contains private information or proprietary information!

##  — Uploading data (Option 2) Connecting to Google Drive

By default, Colab can't access files in your Google Drive. You need to **mount** your Drive first.

**Mounting gives you access to:**
- Your personal Google Drive files and folders
- Shared drives
- Any datasets or files you've saved there

**How to mount Google Drive:**
Run the cell below — you'll be prompted to sign in and grant access.

**Note:** Google requires full access to your Google account. If you don't want them to have this access to your main account, consider using a separate account.

Once mounted, your Drive files are accessible at the path `/content/drive/MyDrive/`

**Note:** make sure you know where newly generated files from this script go - to your Google Drive or just to the 'content' folder which doesn't persist.
